In [1]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader

C:\Workspace\projects\ml\fintech-credit-risk-xai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chroma_client = chromadb.PersistentClient(path="../chroma_db")
collection_name = "pojk_40_2024"

try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass

collection = chroma_client.create_collection(name=collection_name)
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

pdf_path = '../docs/POJK/POJK 40 Tahun 2024 Layanan Pendanaan Bersama Berbasis Teknologi Informasi.pdf'

C:\Workspace\projects\ml\fintech-credit-risk-xai\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|███████████████████████████████████████████████████████

In [3]:
def extract_and_chunk_pdf(filepath, chunk_size=1000, overlap=200):
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [5]:
print("Extracting and splitting PDF documents...")
chunks = extract_and_chunk_pdf(pdf_path)
print(f"Total potongan dokumen (chunks) berhasil dibuat: {len(chunks)}")

print("Converting text to vector and saving it to database... (Please wait)")
documents = []
embeddings = []
metadatas = []
ids = []

for i, chunk in enumerate(chunks):
    documents.append(chunk)
    # Ubah teks ke vektor dimensi tinggi
    vector = embedding_model.encode(chunk).tolist()
    embeddings.append(vector)
    metadatas.append({"source": "POJK_40_2024", "chunk_index": i})
    ids.append(f"chunk_{i}")

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"--- PEMBANGUNAN VECTOR DB SELESAI ---")
print(f"Database tersimpan di: {os.path.abspath('../chroma_db')}")
print(f"Total Dokumen di Database: {collection.count()}")

query = "Bagaimana aturan tentang analisis kelayakan kredit dan kemampuan membayar nasabah (credit scoring)?"
query_vector = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_vector],
    n_results=2
)

print("\n--- RETRIEVAL TEST RESULTS ---")
for i, res in enumerate(results['documents'][0]):
    print(f"\n[Relevansi ke-{i+1}]:\n{res[:500]}...")

Extracting and splitting PDF documents...
Total potongan dokumen (chunks) berhasil dibuat: 63
Converting text to vector and saving it to database... (Please wait)
--- PEMBANGUNAN VECTOR DB SELESAI ---
Database tersimpan di: C:\Workspace\projects\ml\fintech-credit-risk-xai\chroma_db
Total Dokumen di Database: 63

--- RETRIEVAL TEST RESULTS ---

[Relevansi ke-1]:
oleh Penyelenggara ditetapkan oleh Otoritas Jasa Keuangan. Pasal 149 (1) Penyelenggara wajib menerapkan mitigasi risiko penyaluran Pendanaan dengan memperhatikan: a. batas minimum usia calon Penerima Dana; dan b. batas minimum penghasilan calon Penerima Dana. (2) Otoritas Jasa Keuangan dapat menetapkan batas minimum usia calon Penerima Dan a dan batas minimum penghasilan calon Penerima Dana sebagaimana dimaksud pada ayat (1). (3) Ketentuan mengenai batas minimum usia dan batas minimum penghasila...

[Relevansi ke-2]:
Umum Pemberi Dana yang telah ditetapkan. Bagian Keenam Audit Internal Pasal 200 (1) Penyelenggara wajib memiliki 